> Implementation update: the production model library, bounded hybrid adaptation and empirical interval-calibration APIs are now implemented. See [HYBRID_AND_UNCERTAINTY.md](HYBRID_AND_UNCERTAINTY.md) and [notebook 03](03_hybrid_adaptation_and_uncertainty.ipynb). The numerical lessons below are baseline examples, not customer validation.

# 01 · Understand a jacketed chemical reactor

**Goal:** explain why the same reaction recipe can behave differently in receiving equipment.

This is an authored synthetic learning exercise. It uses the actual new backend module, not a second notebook-only implementation. It is not experimental validation, a qualified plant twin or a process safety study.

Work through one section at a time. Finish by writing your own explanation of the transfer risk.


In [ ]:
from pathlib import Path
import sys
import json
from copy import deepcopy

root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "backend/src/lifesci/chemical_reactor/model.py").exists()), None)
if root is None:
    raise RuntimeError("Open this notebook inside a BioDemo checkout containing the chemical_reactor module")
sys.path.insert(0, str(root / "backend"))
from src.lifesci.chemical_reactor.demo import example, receiver_example, synthetic_observations
from src.lifesci.chemical_reactor.model import simulate
from src.lifesci.chemical_reactor.analysis import compare_sites, fit_parameters, sensitivity
import matplotlib.pyplot as plt

source = example()
print(json.dumps(source, indent=2))


## 1. Derive the material balance

Let A be starting material, B desired product and I impurity. Our illustrative reactions are $A\rightarrow B$ and $A\rightarrow I$.

The physical rule is **accumulation = in − out + generation**. A batch has no material inlet or outlet in this model. Consequently:

$$\frac{dN_A}{dt}=-r_B-r_I,\qquad \frac{dN_B}{dt}=r_B,\qquad \frac{dN_I}{dt}=r_I.$$

We **assume**, rather than derive from conservation, that each rate is first order: $r_B=k_B(T)N_A$ and $r_I=k_I(T)N_A$. The rates have units mol/s. Kinetic experiments are needed to justify this assumption for a real chemistry.

The 1:1 reactions conserve $N_A+N_B+N_I$. This is an A-equivalent mole balance. Real chemical mass conservation also needs molecular weights and full stoichiometry.

**Check yourself:** if twice as much A is present at the same temperature and volume-independent kinetics, why does the assumed reaction rate double?


## 2. Derive the energy balance

With no feed, evaporation, phase change or shaft-work contribution:

$$\rho V C_p\frac{dT}{dt}=(-\Delta H_B)r_B+(-\Delta H_I)r_I-UA_{ht}(T-T_c).$$

- Left: how fast stored thermal energy changes, in watts.
- First two right terms: heat released by each reaction; negative reaction enthalpy releases heat.
- Last term: heat removed through the jacket boundary.
- $U$ is an effective heat-transfer coefficient, $A_{ht}$ the wetted heat-transfer area and $T_c$ prescribed coolant temperature.

Both kinetics and heat transfer matter. We assume Arrhenius temperature dependence:

$$k(T)=k_{ref}\exp\left[-\frac{E_a}{R}\left(\frac1T-\frac1{T_{ref}}\right)\right].$$

The example impurity pathway has the higher activation energy. Its rate therefore increases more strongly with increasing temperature over these conditions.

**Check yourself:** if volume grows faster than heat-transfer area, what happens to heat removal per unit volume? Why must real U and utility capacity still be measured?


In [ ]:
donor = simulate(source)
receiver_config = receiver_example(source)
comparison = compare_sites(source, receiver_config, allowed_yield_loss_fraction=0.02)
receiver = comparison["receiver"]
for label, result in [("donor", donor), ("receiver", receiver)]:
    print(label, {k: result[k] for k in ["status", "product_yield_fraction", "impurity_fraction", "violations"]})
print("Transfer screen:", comparison["assessment"])


## 3. See why conversion and desired yield differ

Plot only the trajectory the solver actually completed. A temperature-limit event stops the run; a partial trajectory is not a final-yield prediction. The peak shown is over returned numerical points, not a guarantee about a physical hot spot.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for label, result in [("10 L donor", donor), ("100 L receiver", receiver)]:
    rows = result["trajectory"]
    hours = [r["time_s"]/3600 for r in rows]
    axes[0].plot(hours, [r["temperature_k"]-273.15 for r in rows], label=label)
    axes[1].plot(hours, [100*r["product_yield_fraction"] for r in rows], label=label)
    axes[2].plot(hours, [100*r["impurity_fraction"] for r in rows], label=label)
axes[0].axhline(source["recipe"]["temperature_limit_k"]-273.15, ls="--", color="black")
for ax, ylabel in zip(axes, ["Temperature (°C)", "Desired-product yield (%)", "Impurity (% of initial A)"]):
    ax.set(xlabel="Time (h)", ylabel=ylabel)
    ax.grid(alpha=.2)
axes[0].legend()
fig.suptitle("Synthetic assumptions · no experimental validation")
fig.tight_layout()
plt.show()


## 4. Change the site, one hypothesis at a time

Try coolant temperature, wetted area or effective U. U cannot be freely commanded: changing it here describes an equipment/characterisation hypothesis. Real utility limits and mixing must be checked separately.

The tolerance below means **two absolute yield percentage points**. It is an illustrative task criterion, not a commercial acceptance threshold.


In [ ]:
candidate = deepcopy(receiver_config)
candidate["equipment"]["heat_transfer_coefficient_w_m2_k"] = 450.0
screen = compare_sites(source, candidate, allowed_yield_loss_fraction=0.02)
print({k: screen[k] for k in ["assessment", "yield_loss_fraction_points"]})
print("Receiver constraints:", screen["receiver"]["violations"])
# Exercise: increase effective area instead. Explain the difference between
# a feasible model scenario and equipment that is actually available.


## 5. Fit U from observations

The observations below are generated by this same model with seeded temperature noise. They are useful for understanding calibration, but are **not independent validation**.

Only the first hour enters parameter fitting. Later observations measure predictive discrepancy conditional on the assumed chemistry. Product assays would be required to validate desired-product yield.


In [ ]:
observations = synthetic_observations(source)
starting_guess = deepcopy(source)
starting_guess["equipment"]["heat_transfer_coefficient_w_m2_k"] = 280.0
fit = fit_parameters(starting_guess, observations,
                     fields=["heat_transfer_coefficient_w_m2_k"],
                     bounds={"heat_transfer_coefficient_w_m2_k": [200.0, 600.0]},
                     training_until_s=3600.0)
print({k: fit[k] for k in ["estimates", "training_rmse_k", "holdout_rmse_k",
                           "identifiability_diagnostic", "at_bound"]})


## 6. Find what matters and decide what to measure

This is one-at-a-time sensitivity under declared perturbations—not calibrated uncertainty or an optimal experiment design. A failed scenario stays visible.

Consider measuring effective U under representative load, reaction kinetics at several temperatures, heats of reaction and time-resolved product/impurity concentrations. Decide which question each measurement resolves.


In [ ]:
checks = sensitivity(source, fractional_change=0.1)
for row in checks["rows"]:
    print(row)


## 7. Your tasks before adding another model

1. Set reaction enthalpies to zero and coolant equal to initial temperature. Explain why temperature stays constant.
2. Set U to zero. Compare heat released with stored thermal energy; do not relax a real plant limit to do this experiment.
3. Change only holdout observations. Confirm U does not change but holdout error does.
4. Explain why the same fitted U cannot simply be copied to another vessel.
5. Explain why a good temperature fit does not prove yield prediction accuracy.
6. Write a five-sentence transfer assessment: evidence, prediction, limitation, missing measurement and next experiment.

**Then:** add semi-batch feed/volume balances, followed by crystallisation and isolation models. See [the learning and customer-readiness roadmap](README.md).

The hybrid paper comes later: replace a demonstrably inadequate rate relationship with a constrained learned correction, compare against ordinary refitting, and validate using independent receiver data.
